In [143]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

In [144]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
613,"The movie actually has a fairly good story, bu...",positive
652,"Anyone who thinks Kool Moe Dee, Carol Alt, and...",negative
40,"Thanks Jymn Magon, for creating Disney's 2 bes...",positive
976,This early film has its flaws-- a predictable ...,positive
1,I switched this on (from cable) on a whim and ...,positive


In [145]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from teh text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f"Error duting text normaliztion: {e}")
        raise

In [146]:
df = normalize_text(df)
df.head()

,review,sentiment
613,movie actually fairly good story get bogged se...,positive
652,anyone think kool moe dee carol alt corey feld...,negative
40,thanks jymn magon creating disney s best carto...,positive
976,early film flaw predictable plot overlong scen...,positive
1,switched from cable whim treated quite surpris...,positive


In [147]:
df['sentiment'].value_counts()

sentiment
negative    254
positive    246
Name: count, dtype: int64

In [148]:
x = df["sentiment"].isin(['positive', 'negative'])
df = df[x]

In [149]:
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})
df.head()

,review,sentiment
613,movie actually fairly good story get bogged se...,1
652,anyone think kool moe dee carol alt corey feld...,0
40,thanks jymn magon creating disney s best carto...,1
976,early film flaw predictable plot overlong scen...,1
1,switched from cable whim treated quite surpris...,1


In [150]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [158]:
vectorizer = CountVectorizer(max_features=100)
X = vectorizer.fit_transform(df['review'])
y = df["sentiment"]

In [159]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [160]:
import dagshub
from dotenv import load_dotenv

load_dotenv()
import os

mlflow.set_tracking_uri(os.environ.get("mlflow_tracking_uri"))
dagshub.init(
    repo_owner=os.environ.get("mlflow_repo_owner"),
    repo_name=os.environ.get("mlflow_repo_name"),
    mlflow=True)

mlflow.set_experiment("Logistic Regression Baseline")

2026-07-12 22:47:35,658 - INFO - HTTP Request: GET https://dagshub.com/api/v1/repos/Kanak10/Capstone-Project "HTTP/1.1 200 OK"


Initialized MLflow to track repo "Kanak10/Capstone-Project"

2026-07-12 22:47:35,663 - INFO - Initialized MLflow to track repo "Kanak10/Capstone-Project"


Repository Kanak10/Capstone-Project initialized!

2026-07-12 22:47:35,665 - INFO - Repository Kanak10/Capstone-Project initialized!


<Experiment: artifact_location='mlflow-artifacts:/d5766c04240b470d9dbed2a4139537b4', creation_time=1783866603033, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1783866603033, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}, trace_location=None, workspace='default'>

In [164]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()

    try:
        logging.info("Logging preprocessing parametes...")
        mlflow.log_param("vectorizer", "Bag of words")
        mlflow.log_param("num_features", 100)
        mlflow.log_param("test_size", 0.25)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000) # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parametes...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and loggign completed in {end_time - start_time:.2f} seconds.")

        # save and log notebook
        notebook_path = "exp1.ipynb"
        logging.info("Executing Jupyter Notebook. This may take a while...")
        os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        mlflow.log_artifact(notebook_path)

        logging.info("Notebook execution and logging complete.")

        # Pring the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")
    except Exception as e:
        logging.error(f"An error occured: {e}", exc_info=True)

2026-07-12 22:53:25,374 - INFO - Starting MLflow run...
2026-07-12 22:53:25,848 - INFO - Logging preprocessing parametes...
2026-07-12 22:53:26,951 - INFO - Initializing Logistic Regression model...
2026-07-12 22:53:26,953 - INFO - Fitting the model...
2026-07-12 22:53:26,985 - INFO - Model training complete.
2026-07-12 22:53:26,986 - INFO - Logging model parametes...
2026-07-12 22:53:27,296 - INFO - Making predictions...
2026-07-12 22:53:27,297 - INFO - Calculating evaluation metrics...
2026-07-12 22:53:27,304 - INFO - Logging evaluation metrics...
2026-07-12 22:53:28,795 - INFO - Saving and logging the model...
2026/07/12 22:53:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/12 22:53:37 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026-07-12 22:53:44,565 - INFO - Model training and loggign completed in 18.72 seconds.
2026-07

🏃 View run secretive-toad-996 at: https://dagshub.com/Kanak10/capstone_project.mlflow/#/experiments/0/runs/3277ac0dc1d9455a8f924f8887994ba8
🧪 View experiment at: https://dagshub.com/Kanak10/capstone_project.mlflow/#/experiments/0
